In [1]:
12

12

# 원핫인코딩 (One-hot Encoding)

- 의미: 단어(또는 토큰)를 어휘 크기(vocab size)만큼의 길이를 가진 벡터로 바꾸는 방법
- 표현 방식: 해당 단어의 인덱스 위치만 1, 나머지는 전부 0 (즉, “이 단어가 맞다/아니다”만 표시)
- 예시: vocab size가 5이고 단어 인덱스가 3이면 → [0, 0, 1, 0, 0]
- 특징/주의: 단어 간 의미/유사도 정보는 없고, vocab이 커질수록 벡터가 매우 커져 메모리/연산 비용이 증가함
    - 그래서 실무에선 보통 임베딩(Embedding)으로 저차원 밀집(dense) 벡터로 바꿔 사용한다
- 사용 시기: 단어 수가 아주 작거나(카테고리/키워드 몇십~몇백) 간단한 모델/해석이 중요한 경우엔 원-핫/BoW를 사용할 때도 있다.

- 임베딩
    - 학습 가능한 변환(레이어/행렬): 단어 인덱스 k를 길이 d(예: 64, 128)인 밀집(dense) 벡터로 매핑
    - 예: k=3 → [0.12, -0.03, ...] (d차원)
    - 특징: 저차원, 밀집, 학습을 통해 의미/유사도가 반영될 수 있음

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.
The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.
On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.
Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.
The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things.
"""

sentences = sent_tokenize(raw_text)  # 문장 단위로 분리

en_stopwords = stopwords.words('english')  # 영어 불용어 목록

vocab = {}  # key:해당단어, value: 빈도수
preprocessed_sentences = []  # 전처리된 문장

for sentence in sentences:
    sentence = sentence.lower()       # 소문자 통일
    tokens = word_tokenize(sentence)  # 문장 -> 단어 토큰화
    tokens = [token for token in tokens if token not in en_stopwords]  # 불용어 제거
    tokens = [token for token in tokens if len(token) > 2]  # 길이 2 이하 제거

    for token in tokens:
        if token not in vocab:
            vocab[token] = 1   # 없으면 키, 값 1 추가
        else:
            vocab[token] += 1  # 있으면 값 +1 추가

    preprocessed_sentences.append(tokens)  # 전처리된 토큰 결과

print(preprocessed_sentences)

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 상위 15개만 사용 (필터링용), 그 외 토큰은 OOV 처리
tokenizer = Tokenizer(num_words=15, oov_token='<OOV>')

# 단어 빈도 기반 인덱스 사전
tokenizer.fit_on_texts(preprocessed_sentences)    # 코퍼스에서 단어 빈도 기반 인덱스 사전 생성
# 이후로 word_index나 index_word를 확인가능

# 문장(토큰 리스트)을 정수 인덱스 시퀀스로 변환
sequences = tokenizer.texts_to_sequences(preprocessed_sentences)

# 패딩 처리 (10보다 길면 앞에서 자르고, 10보다 짧으면 앞에서 패딩 토큰 붙임)
padded = pad_sequences(sequences, maxlen=10, truncating='pre')

print(padded.shape, padded)

NameError: name 'preprocessed_sentences' is not defined

In [ ]:
# 정수 라벨/인덱스를 원-핫 벡터로 바꿔주는 함수
from tensorflow.keras.utils import to_categorical

# (문장, 길이) 정수 인덱스 -> (문장, 길이, 클래스 수) 원-핫으로 변환
one_hot_encodded = to_categorical(padded)
print(one_hot_encodded, one_hot_encodded.shape)

### 한국어 전처리
1. 토큰화 (형태소 분석)
2. 시퀀스 처리 
3. 패딩 처리 
4. one-hot encoding

In [ ]:
texts = [
    '나는 오늘 학원에 간다.',
    '친구들과 맛있는 점심 식사를 했다.',
    '오늘은 어떤 즐거운 수업을 할지 너무 기대가 된다.'
]

In [ ]:
from konlpy.tag import Okt
import re

okt = Okt()

def load_stopwords(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        stopwords = [line.strip() for line in f]  # 줄바꿈/공백 제거하여 리스트 생성
    return stopwords

ko_stopwords = load_stopwords('ko_stopwords.txt')

preprocessed_texts = []

for text in texts:
    tokens = okt.morphs(text, stem=True)  # 형태소 단위로 분리 (어간추출)
    tokens = [token for token in tokens if token not in ko_stopwords]  # 불용어 제거
    # 공백, 구두점, 기호가 포함된 토큰 제거
    tokens = [token for token in tokens if not re.search(r'[\s.,:;?!]', token)]
    preprocessed_texts.append(tokens)

print(preprocessed_texts)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 사전에 없는 토큰은 OOV 처리
tokenizer = Tokenizer(oov_token='<OOV>')

# 단어 빈도 기반 인덱스 사전
tokenizer.fit_on_texts(preprocessed_texts)    # 코퍼스에서 단어 빈도 기반 인덱스 사전 생성
# 이후로 word_index나 index_word를 확인가능

# 문장(토큰 리스트)을 정수 인덱스 시퀀스로 변환
sequences = tokenizer.texts_to_sequences(preprocessed_texts)
sequences

AssertionError: phrase input should be string, not <class 'list'>

In [ ]:
tokenizer.word_index  # 인덱스가 작을수록 더 자주 등장 (OOV 제외)

NameError: name 'tokens' is not defined

In [ ]:
# 시퀀스 패딩/트런케이팅 유틸 함수
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded = pad_sequences(sequences, maxlen=9)  # 기본 설정(pre padding, maxlen=최장 길이, pad value=0)

print(padded, padded.shape)

In [ ]:
# 정수 라벨/인덱스를 원-핫 벡터로 바꿔주는 함수
from tensorflow.keras.utils import to_categorical

# (문장, 길이) 정수 인덱스 -> (문장, 길이, 클래스 수) 원-핫으로 변환
one_hot_encodded = to_categorical(padded)
print(one_hot_encodded, one_hot_encodded.shape)

shape (3, 9, 18)
-> (문장, 길이, 클래스 수)
-> 클래스 수 = 최대인덱스 +1 (pad 추가)

In [ ]:
from tensorflow.keras import models, layers  # 모델 레이어

input = layers.Input(shape=(9, 18))  # 입력 텐서 (timesteps=9, features=18)
x = layers.SimpleRNN(8)(input)       # RNN 은닉유닛 8개 -> 마지막 은닉상태로 전송
output = layers.Dense(1, activation='sigmoid')(x)  # 이진분류 출력 : 확률 0~1 1개로 변환

model = models.Model(inputs=input, outputs=output)  # API 형태로 입력 -> 출력 연결해서 모델 생성
model.summary()
# 입력 (None, 3, 19) 형태로 받아, SimpleRNN(8)을 거치면 (None, 8) -> Dense(1)로 (None, 1) 확률 출력

In [ ]:
import numpy as np

# 설정 (손실함수/최적화함수/평가지표)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

labels = np.array([1, 1, 1])  # 샘플 문장들에 대한 정답

# 학습 
model.fit(one_hot_encodded, labels, epochs=10)

임베딩 사용 전 주로 사용하던 기법인 원-핫 인코딩 기법을 사용하였다.

- 원핫 인코딩은 vocab 차원이 커지면 커질수록 메모리/연산이 매우 비효율적
- 단어 의미/유사도 정보도 제대로 반영하지 못함
=> 이러한 이유로 정수 인덱스를 그대로 넣는 Embedding기법을 활용

- 입력 (batch, timesteps) 정수 인덱스
-> Embedding(vocab_size, embed_dim) -> (batch, timesteps, embed_dim)
-> 이렇게 만든 후에 RNN/Transformer 모델로 학습 진행

| 표현 방식 | 입력 단위 | 벡터 형태(Shape) | 값이 의미하는 것 | 순서(문맥) 보존 | 장점 | 단점 | 주 사용처 |
|---|---|---|---|---|---|---|---|
| **원-핫(시퀀스)** | 토큰 시퀀스(문장) | `(seq_len, V)` 또는 `(batch, seq_len, V)` | 각 토큰을 길이 `V` 벡터로 표현하며 해당 인덱스만 1 | ✅ | 구현과 이해가 쉬움, RNN 입력으로 바로 사용 가능 | `V`가 커지면 메모리·연산량 증가, 의미·유사도 정보 없음 | 교육용 데모, 작은 vocabulary 실험 |
| **BoW (CountVectorizer)** | 문서(문장/리뷰) 1개 | `(V)` 또는 `(batch, V)` | 단어의 등장 횟수(count) | ❌ | 빠르고 간단하며 전통 ML에서 활용하기 좋음 | 단어 순서와 문맥 손실, 고차원 희소 벡터 | 스팸 분류, 감성 분석 베이스라인, 빠른 EDA |
| **TF-IDF** | 문서(문장/리뷰) 1개 | `(V)` 또는 `(batch, V)` | 단어 중요도 = TF × IDF | ❌ | BoW보다 중요한 단어를 잘 반영, 전통 ML에서 성능이 좋은 편 | 문맥과 순서 손실, 고차원 희소 벡터 | 검색, 문서 유사도, Linear SVM·로지스틱 회귀 기반 분류 |
| **Embedding** | 토큰 시퀀스(문장) | `(seq_len, d)` 또는 `(batch, seq_len, d)` | 토큰 ID를 `d`차원의 실수 벡터로 변환 | ✅ | 저차원 밀집 벡터, 의미와 유사도 표현 가능, 딥러닝에서 널리 사용 | 학습 데이터·자원 필요, 해석이 상대적으로 어려움 | RNN, LSTM, GRU, Transformer 등 딥러닝 NLP 모델 입력 |